In [1]:
print("Isha")

Isha


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


class SalesDataAnalyzer:
    def __init__(self):
        self.data = None
        self.file_name = ""
        sns.set_theme(style="whitegrid")

    def __del__(self):
        plt.close("all")

    def check_data(self):
        if self.data is None:
            print("Please load a dataset first.")
            return False
        return True

    def load_data(self):
        path = input("Enter CSV file name or path: ").strip().strip('"')
        if not os.path.exists(path):
            print("File not found. Check the file path and try again.")
            return
        try:
            data = pd.read_csv(path)
            data.columns = data.columns.astype(str).str.strip()
            self.data = data
            self.file_name = path
            print("Dataset loaded successfully!")
            print("Rows:", len(self.data))
            print("Columns:", len(self.data.columns))
        except Exception as error:
            print("Error loading file:", error)

    def explore_data(self):
        if not self.check_data():
            return

        while True:
            print("\n== Explore Data ==")
            print("1. Display the first 5 rows")
            print("2. Display the last 5 rows")
            print("3. Display column names")
            print("4. Display data types")
            print("5. Display basic information")
            print("6. Return to main menu")

            choice = input("Enter your choice: ").strip()

            if choice == "1":
                print("\nFirst 5 rows:\n")
                print(self.data.head())

            elif choice == "2":
                print("\nLast 5 rows:\n")
                print(self.data.tail())

            elif choice == "3":
                print("\nColumn names:\n")
                print(self.data.columns.tolist())

            elif choice == "4":
                print("\nData types:\n")
                print(self.data.dtypes)

            elif choice == "5":
                print("\nBasic information:\n")
                self.data.info()

            elif choice == "6":
                print("Returning to main menu...")
                break

            else:
                print("Invalid choice. Enter a number from 1 to 6")

    def clean_data(self):
        if not self.check_data():
            return
        self.data.columns = (
            self.data.columns.astype(str).str.strip().str.replace(" ", "_", regex=False)
        )
        for column in self.data.columns:
            if "date" in column.lower():
                self.data[column] = pd.to_datetime(self.data[column], errors="coerce")
        print("Column names cleaned and date columns converted where possible.")

    def handle_missing_data(self):
        if not self.check_data():
            return
        while True:
            print("\n=== Handle Missing Data ===")
            print("1. Display rows with missing values")
            print("2. Fill numeric missing values with mean")
            print("3. Drop rows with missing values")
            print("4. Replace missing values with a specific value")
            print("5. Return to main menu")
            choice = input("Enter your choice: ").strip()

            if choice == "1":
                missing_rows = self.data[self.data.isna().any(axis=1)]
                if missing_rows.empty:
                    print("No missing values found in the dataset!")
                else:
                    print(missing_rows)
            elif choice == "2":
                numeric_columns = self.data.select_dtypes(include=np.number).columns
                if len(numeric_columns) == 0:
                    print("No numeric columns found.")
                else:
                    self.data[numeric_columns] = self.data[numeric_columns].fillna(
                        self.data[numeric_columns].mean()
                    )
                    print("Missing numeric values filled with the mean.")
            elif choice == "3":
                before = len(self.data)
                self.data = self.data.dropna()
                print("Removed", before - len(self.data), "rows.")
            elif choice == "4":
                value = input("Enter replacement value: ")
                self.data = self.data.fillna(value)
                print("Missing values replaced.")
            elif choice == "5":
                break
            else:
                print("Invalid choice. Enter 1 to 5.")

    def choose_column(self, columns, message):
        columns = list(columns)
        if not columns:
            print("No suitable columns found.")
            return None
        print("\nAvailable columns:")
        for number, column in enumerate(columns, start=1):
            print(f"{number}. {column}")
        try:
            choice = int(input(message))
            if 1 <= choice <= len(columns):
                return columns[choice - 1]
        except ValueError:
            pass
        print("Invalid selection.")
        return None

    def choose_numeric_column(self):
        if not self.check_data():
            return None
        columns = self.data.select_dtypes(include=np.number).columns.tolist()
        return self.choose_column(columns, "Select a numeric column: ")

    def choose_category_column(self):
        if not self.check_data():
            return None
        columns = self.data.select_dtypes(
            include=["str", "category", "bool"]
        ).columns.tolist()
        return self.choose_column(columns, "Select a category column: ")

    def mathematical_operations(self):
        if not self.check_data():
            return
        column = self.choose_numeric_column()
        if column is None:
            return
        values = self.data[column].dropna().to_numpy()
        print("First values:", values[:10])
        print("Add 2:", values[:10] + 2)
        print("Multiply by 2:", values[:10] * 2)
        print("Divide by 2:", values[:10] / 2)
        print("Sorted values:", np.sort(values)[:10])
        print("Square root (non-negative values only):", np.sqrt(values[values >= 0][:10]))

    def combine_and_split_arrays(self):
        if not self.check_data():
            return
        numeric = self.data.select_dtypes(include=np.number)
        if numeric.shape[1] < 2:
            print("At least two numeric columns are required.")
            return
        first = numeric.iloc[:, 0].to_numpy()
        second = numeric.iloc[:, 1].to_numpy()
        combined = np.column_stack((first, second))
        print("Combined NumPy array (first 5 rows):\n", combined[:5])
        parts = np.array_split(combined, 2)
        print("First split:\n", parts[0][:5])
        print("Second split:\n", parts[1][:5])

    def search_sort_filter(self):
        if not self.check_data():
            return
        column = self.choose_column(self.data.columns, "Select a column: ")
        if column is None:
            return
        print("1. Search")
        print("2. Sort ascending")
        print("3. Filter numeric values greater than a number")
        choice = input("Choose an option: ").strip()
        if choice == "1":
            value = input("Enter search text: ")
            result = self.data[self.data[column].astype(str).str.contains(
                value, case=False, na=False
            )]
            print(result.head(20))
        elif choice == "2":
            try:
                result = self.data.sort_values(by=column)
                print(result.head(20))
            except Exception as error:
                print("Could not sort this column:", error)
        elif choice == "3":
            if not pd.api.types.is_numeric_dtype(self.data[column]):
                print("Please choose a numeric column.")
                return
            try:
                value = float(input("Show values greater than: "))
                print(self.data[self.data[column] > value].head(20))
            except ValueError:
                print("Please enter a valid number.")
        else:
            print("Invalid choice.")

    def aggregate_functions(self):
        if not self.check_data():
            return
        numeric = self.data.select_dtypes(include=np.number)
        if numeric.empty:
            print("No numeric columns found.")
            return
        print(numeric.agg(["count", "sum", "mean", "min", "max", "std"]))

    def statistical_analysis(self):
        if not self.check_data():
            return
        numeric = self.data.select_dtypes(include=np.number)
        if numeric.empty:
            print("No numeric columns found.")
            return
        print("\nDescriptive statistics:\n", numeric.describe())
        print("\nVariance:\n", numeric.var())
        print("\nStandard deviation:\n", numeric.std())
        print("\n25th, 50th, and 75th percentiles:\n", numeric.quantile([0.25, 0.50, 0.75]))
        print("\nCorrelation:\n", numeric.corr())

    def create_pivot_table(self):
        if not self.check_data():
            return
        group_col = self.choose_column(self.data.columns, "Select a grouping column: ")
        value_col = self.choose_numeric_column()
        if group_col is None or value_col is None:
            return
        table = pd.pivot_table(
            self.data, index=group_col, values=value_col, aggfunc="sum"
        )
        print("\nPivot table:\n", table)

    def groupby_transform(self):
        if not self.check_data():
            return
        group_col = self.choose_column(self.data.columns, "Select a group column: ")
        value_col = self.choose_numeric_column()
        if group_col is None or value_col is None:
            return
        result = self.data[[group_col, value_col]].copy()
        result["Group_Mean"] = result.groupby(group_col)[value_col].transform("mean")
        print(result.head(20))

    def rename_column(self):
        if not self.check_data():
            return
        print("Columns:", self.data.columns.tolist())
        old_name = input("Enter the existing column name: ").strip()
        if old_name not in self.data.columns:
            print("Column not found.")
            return
        new_name = input("Enter the new column name: ").strip()
        if not new_name:
            print("New column name cannot be empty.")
            return
        self.data = self.data.rename(columns={old_name: new_name})
        print("Column renamed successfully.")

    def dataframe_operations(self):
        if not self.check_data():
            return
        while True:
            print("\n=== DataFrame Operations ===")
            print("1. Mathematical operations")
            print("2. Combine and split NumPy arrays")
            print("3. Search, sort, and filter")
            print("4. Aggregation functions")
            print("5. Statistical analysis")
            print("6. Create pivot table")
            print("7. GroupBy and transform")
            print("8. Rename a column")
            print("9. Clean column names and date columns")
            print("10. Return to main menu")
            choice = input("Enter your choice: ").strip()
            if choice == "1":
                self.mathematical_operations()
            elif choice == "2":
                self.combine_and_split_arrays()
            elif choice == "3":
                self.search_sort_filter()
            elif choice == "4":
                self.aggregate_functions()
            elif choice == "5":
                self.statistical_analysis()
            elif choice == "6":
                self.create_pivot_table()
            elif choice == "7":
                self.groupby_transform()
            elif choice == "8":
                self.rename_column()
            elif choice == "9":
                self.clean_data()
            elif choice == "10":
                break
            else:
                print("Invalid choice. Enter 1 to 10.")

    def bar_plot(self):
        category = self.choose_category_column()
        value = self.choose_numeric_column()
        if category is None or value is None:
            return
        result = self.data.groupby(category, dropna=False)[value].sum().sort_values(
            ascending=False
        ).head(15)
        result.plot(kind="bar", title=f"{value} by {category}")
        plt.xlabel(category)
        plt.ylabel(value)
        plt.tight_layout()
        plt.show()

    def line_plot(self):
        if not self.check_data():
            return
        value = self.choose_numeric_column()
        if value is None:
            return
        self.data[value].reset_index(drop=True).plot(
            kind="line", title=f"{value} by row"
        )
        plt.xlabel("Row")
        plt.ylabel(value)
        plt.tight_layout()
        plt.show()

    def scatter_plot(self):
        if not self.check_data():
            return
        columns = self.data.select_dtypes(include=np.number).columns.tolist()
        x_col = self.choose_column(columns, "Select X column: ")
        y_col = self.choose_column(columns, "Select Y column: ")
        if x_col is None or y_col is None:
            return
        sns.scatterplot(data=self.data, x=x_col, y=y_col)
        plt.title(f"{y_col} versus {x_col}")
        plt.tight_layout()
        plt.show()

    def pie_chart(self):
        category = self.choose_category_column()
        value = self.choose_numeric_column()
        if category is None or value is None:
            return
        result = self.data.groupby(category)[value].sum().nlargest(8)
        if result.empty or result.sum() == 0:
            print("No suitable values for a pie chart.")
            return
        result.plot(kind="pie", autopct="%1.1f%%", title=f"{value} by {category}")
        plt.ylabel("")
        plt.tight_layout()
        plt.show()

    def histogram(self):
        if not self.check_data():
            return
        column = self.choose_numeric_column()
        if column is None:
            return
        sns.histplot(data=self.data, x=column, bins=20, kde=True)
        plt.title(f"Distribution of {column}")
        plt.tight_layout()
        plt.show()

    def stack_plot(self):
        if not self.check_data():
            return
        numeric = self.data.select_dtypes(include=np.number)
        if numeric.empty:
            print("No numeric columns found.")
            return
        numeric.head(30).plot(kind="area", stacked=False, title="Stacked numeric data")
        plt.xlabel("Row")
        plt.tight_layout()
        plt.show()

    def heatmap(self):
        if not self.check_data():
            return
        numeric = self.data.select_dtypes(include=np.number)
        if numeric.shape[1] < 2:
            print("At least two numeric columns are needed for a heatmap.")
            return
        sns.heatmap(numeric.corr(), annot=True, cmap="coolwarm", fmt=".2f")
        plt.title("Correlation Heatmap")
        plt.tight_layout()
        plt.show()

    def box_plot(self):
        if not self.check_data():
            return
        column = self.choose_numeric_column()
        if column is None:
            return
        sns.boxplot(data=self.data, y=column)
        plt.title(f"Box Plot of {column}")
        plt.tight_layout()
        plt.show()

    def visualize_data(self):
        if not self.check_data():
            return
        while True:
            print("\n=== Data Visualization ===")
            print("1. Bar plot")
            print("2. Line plot")
            print("3. Scatter plot")
            print("4. Pie chart")
            print("5. Histogram")
            print("6. Stack/area plot")
            print("7. Seaborn heatmap")
            print("8. Seaborn box plot")
            print("9. Return to main menu")
            choice = input("Enter your choice: ").strip()
            if choice == "1":
                self.bar_plot()
            elif choice == "2":
                self.line_plot()
            elif choice == "3":
                self.scatter_plot()
            elif choice == "4":
                self.pie_chart()
            elif choice == "5":
                self.histogram()
            elif choice == "6":
                self.stack_plot()
            elif choice == "7":
                self.heatmap()
            elif choice == "8":
                self.box_plot()
            elif choice == "9":
                break
            else:
                print("Invalid choice.")

    def save_visualization(self):
        if not self.check_data():
            return
        column = self.choose_numeric_column()
        if column is None:
            return
        self.data[column].dropna().plot(kind="hist", title=f"Histogram of {column}")
        plt.xlabel(column)
        plt.tight_layout()
        file_name = input(
            "Enter image name (example: sales_plot.png): "
        ).strip()
        if not file_name:
            file_name = "sales_plot.png"
        if not file_name.lower().endswith((".png", ".jpg", ".jpeg")):
            file_name += ".png"
        try:
            plt.savefig(file_name, dpi=300, bbox_inches="tight")
            print("Visualization saved successfully as", os.path.abspath(file_name))
        except Exception as error:
            print("Could not save visualization:", error)
        finally:
            plt.close()


    def run(self):
        while True:
            print("\n--------- Data Analysis & Visualization Program -----------")
            print("1. Load Dataset")
            print("2. Explore Data")
            print("3. Perform DataFrame Operations")
            print("4. Handle Missing Data")
            print("5. Generate Descriptive Statistics")
            print("6. Data Visualization")
            print("7. Save Visualization")
            print("8. Exit")
            print("===========================================================")
            choice = input("Enter your choice: ").strip()

            if choice == "1":
                self.load_data()
            elif choice == "2":
                self.explore_data()
            elif choice == "3":
                self.dataframe_operations()
            elif choice == "4":
                self.handle_missing_data()
            elif choice == "5":
                self.statistical_analysis()
            elif choice == "6":
                self.visualize_data()
            elif choice == "7":
                self.save_visualization()
            elif choice == "8":
                print("Exiting program. goodbyee !")
                break
            else:
                print("Invalid choice. Enter a number from 1 to 9.")


if __name__ == "__main__":
    analyzer = SalesDataAnalyzer()
    analyzer.run()
